# Burunov Bot — XTTS v2 на 8 идеальных референсах

**Что нового:** кореш вручную перетранскрибировал 50 сегментов с двойной оценкой (звук + похоже на Бурунова). Из них выбраны **8 с рейтингом 5/5 + 5/5** — идеальные кандидаты.

Прогоняем тестовые фразы через все 8, сравниваем, выбираем лучший.

**GPU:** Colab T4

## 1. Установка XTTS v2 + фикс transformers

In [ ]:
!pip uninstall -y TTS tts 2>&1 | tail -3
!pip install coqui-tts 2>&1 | tail -5
# Фикс версии transformers (Coqui ждёт старые функции)
!pip install --force-reinstall "transformers==4.44.2" 2>&1 | tail -5
!pip install soundfile

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ Включи T4 GPU: Runtime → Change runtime type')

## 2. Скачать 8 идеальных референсов (5/5 + 5/5)

Через `git clone` (один HTTP-запрос, без rate limit).

In [ ]:
import os, shutil

REFS_DIR = '/content/refs'
os.makedirs(REFS_DIR, exist_ok=True)

# 8 идеальных референсов (5/5 + 5/5 от кореша)
BEST_REFS = [
    {
        'id': 'ref_0043',
        'wav': '-211220744_456239510_0043.wav',
        'duration': 8.1,
        'pitch': 97,
        'rating': 5,
        'voice_match': 5,
        'text': 'С одной стороны, я никогда не мечтал даже об этом, не грезил и не ставил никаких целей, что произойдет так.',
        'notes': 'Уже пробовали — кореш подтвердил что идеал',
    },
    {
        'id': 'ref_0035',
        'wav': '-211220744_456239510_0035.wav',
        'duration': 5.6,
        'pitch': 97,
        'rating': 5,
        'voice_match': 5,
        'text': 'Не ясно, но люди подходили, несли мне детей разных возрастов, даже вот таких.',
        'notes': 'Короткий, чистый',
    },
    {
        'id': 'ref_0069',
        'wav': '-211220744_456239510_0069.wav',
        'duration': 5.6,
        'pitch': 84,
        'rating': 5,
        'voice_match': 5,
        'text': 'Екатерине она как-то этому всему посодействовала и вдохновила меня.',
        'notes': 'Короткий, низкий pitch (84Hz) — может быть более ленивый',
    },
    {
        'id': 'ref_0181',
        'wav': '-211220744_456239510_0181.wav',
        'duration': 9.8,
        'pitch': 89,
        'rating': 5,
        'voice_match': 5,
        'text': 'Раньше налил стакан, перезагрузился и все в порядке. То сейчас, вот да, к сожалению, джетлак я ну не могу полететь. На другой конец света, например, там.',
        'notes': 'Про джетлаг, длинный',
    },
    {
        'id': 'ref_0155',
        'wav': '-211220744_456239510_0155.wav',
        'duration': 9.4,
        'pitch': 97,
        'rating': 5,
        'voice_match': 5,
        'text': 'У кого программа? Какая программа? Я так и не понимаю. Мне тоже говорил, что у тебя вот у тебя две шестёрки, у тебя две шестёрки.',
        'notes': 'Про нумерологию, ироничный',
    },
    {
        'id': 'ref_0167',
        'wav': '-211220744_456239510_0167.wav',
        'duration': 9.3,
        'pitch': 91,
        'rating': 5,
        'voice_match': 5,
        'text': 'Я это признаю, конечно, но тут вопросы, как бы когда я это понимаю что-то, вопрос уже к моей нестабильной самооценке и почему я.',
        'notes': 'Про самооценку, рефлексивный',
    },
    {
        'id': 'ref_0114',
        'wav': '-211220744_456239510_0114.wav',
        'duration': 10.9,
        'pitch': 104,
        'rating': 5,
        'voice_match': 5,
        'text': 'Я был растерян. Я не знал, с какой стороны, какие ключи подбирать. Но с девочкой понятно. Разность полов, друзья, все и какое-то уже.',
        'notes': 'Про отношения, есть "друзья" — разговорный',
    },
    {
        'id': 'ref_0034',
        'wav': '-211220744_456239510_0034.wav',
        'duration': 14.7,
        'pitch': 94,
        'rating': 5,
        'voice_match': 5,
        'text': 'Получилось так, что мы попали в фантастический фильм всё было в тумане. Подъёмников не было видно, и на нас вот ехали какие-то кабинки, как инопланетная, как эта история. Как это происходило непонятно, коммуникации, как они там передавали друг другу, по очереди может быть.',
        'notes': 'Длинный, сторителлинг',
    },
]

print('Клонируем репо burunov-voice (один запрос)...')
!rm -rf /content/burunov-voice
!git clone --depth 1 https://github.com/shray77/burunov-voice.git /content/burunov-voice 2>&1 | tail -3

SRC_DIR = '/content/burunov-voice/burunov-only-v2/segments'
for ref in BEST_REFS:
    src = os.path.join(SRC_DIR, ref['wav'])
    dst = os.path.join(REFS_DIR, ref['wav'])
    shutil.copy(src, dst)
    print(f'  ✓ {ref["wav"]} (rating={ref["rating"]}/5, voice={ref["voice_match"]}/5, {ref["duration"]}сек)')

print(f'\n✅ {len(BEST_REFS)} wav скачаны в {REFS_DIR}')

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

for ref in BEST_REFS:
    path = os.path.join(REFS_DIR, ref['wav'])
    audio, sr = sf.read(path)
    print(f'\n>>> {ref["id"]} ({ref["duration"]}сек, pitch={ref["pitch"]}Hz)')
    print(f'    Текст: {ref["text"][:100]}')
    print(f'    Notes: {ref["notes"]}')
    display(Audio(path))

## 3. Загрузка XTTS v2

In [ ]:
from TTS.api import TTS

print('Загрузка XTTS v2...')
tts = TTS(model_name='tts_models/multilingual/multi-dataset/xtts_v2',
          device='cuda' if torch.cuda.is_available() else 'cpu')
print('✅ XTTS v2 готова')

## 4. Тест синтеза через все 8 референсов

Прогоняем 3 тестовые фразы через каждый референс (24 синтеза).
После — ячейка 5 проигрывает сравнение.

In [ ]:
import time
import soundfile as sf

TEST_TEXTS = [
    'Угу, щас, Олег Тарасыч... кофеварку найду.',
    'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
    'Вот ваш кофе, Олег. Не обожгись, бля.',
]

os.makedirs('/content/compare', exist_ok=True)
results = []

for ref in BEST_REFS:
    ref_path = os.path.join(REFS_DIR, ref['wav'])
    print(f'\n========== {ref["id"]} ({ref["duration"]}сек, pitch={ref["pitch"]}Hz) ==========')
    
    for i, test_text in enumerate(TEST_TEXTS):
        out = f'/content/compare/{ref["id"]}_test{i}.wav'
        print(f'\n  → [{i+1}/3] "{test_text[:60]}..."')
        
        t0 = time.time()
        try:
            tts.tts_to_file(
                text=test_text,
                speaker_wav=ref_path,
                language='ru',
                file_path=out,
            )
            dt = time.time() - t0
            audio, sr_out = sf.read(out)
            audio_dur = len(audio) / sr_out
            rtf = dt / audio_dur
            print(f'    ✅ Синтез {dt:.1f}с | Аудио {audio_dur:.1f}с | RTF={rtf:.2f}')
            
            results.append({
                'ref_id': ref['id'],
                'ref_rating': ref['rating'],
                'ref_voice': ref['voice_match'],
                'test_idx': i,
                'test_text': test_text,
                'synth_time': dt,
                'audio_dur': audio_dur,
                'rtf': rtf,
                'path': out,
            })
        except Exception as e:
            print(f'    ❌ {e}')
            import traceback; traceback.print_exc()

print(f'\n✅ Синтезировано {len(results)} аудио')

## 5. Сравнение на фразе «Угу, щас, Олег Тарасыч»

In [ ]:
TEST_IDX = 0
print(f'\n>>> Тестовая фраза: "{TEST_TEXTS[TEST_IDX]}"\n')

for ref in BEST_REFS:
    r = next((x for x in results if x['ref_id']==ref['id'] and x['test_idx']==TEST_IDX), None)
    if r is None: continue
    print(f'\n>>> {ref["id"]} (rating={ref["rating"]}/5, voice={ref["voice_match"]}/5, RTF={r["rtf"]:.2f})')
    print(f'    Ref: {ref["text"][:80]}')
    display(Audio(r['path']))

## 6. Сравнение на анекдоте Штирлиц

In [ ]:
TEST_IDX = 1
print(f'\n>>> Тестовая фраза: "{TEST_TEXTS[TEST_IDX]}"\n')

for ref in BEST_REFS:
    r = next((x for x in results if x['ref_id']==ref['id'] and x['test_idx']==TEST_IDX), None)
    if r is None: continue
    print(f'\n>>> {ref["id"]} (rating={ref["rating"]}/5, voice={ref["voice_match"]}/5, RTF={r["rtf"]:.2f})')
    display(Audio(r['path']))

## 7. Сравнение на финальной фразе кофе

In [ ]:
TEST_IDX = 2
print(f'\n>>> Тестовая фраза: "{TEST_TEXTS[TEST_IDX]}"\n')

for ref in BEST_REFS:
    r = next((x for x in results if x['ref_id']==ref['id'] and x['test_idx']==TEST_IDX), None)
    if r is None: continue
    print(f'\n>>> {ref["id"]} (rating={ref["rating"]}/5, voice={ref["voice_match"]}/5, RTF={r["rtf"]:.2f})')
    display(Audio(r['path']))

## 8. Выбери лучший референс

После прослушивания 3 сравнений — впиши какой референс даёт лучший клон Бурунова.

In [ ]:
# ⚠️ ПОМЕНЯЙ ПОСЛЕ ПРОСЛУШИВАНИЯ
# Варианты: ref_0043, ref_0035, ref_0069, ref_0181, ref_0155, ref_0167, ref_0114, ref_0034
BEST_REF_ID = 'ref_0043'  # по умолчанию — попробуй другие!

best_ref = next(r for r in BEST_REFS if r['id']==BEST_REF_ID)
BEST_REF_AUDIO = os.path.join(REFS_DIR, best_ref['wav'])
BEST_REF_TEXT = best_ref['text']

print(f'✅ Выбран референс: {BEST_REF_ID}')
print(f'   Файл: {BEST_REF_AUDIO}')
print(f'   Текст: "{BEST_REF_TEXT}"')
print(f'   Rating: {best_ref["rating"]}/5, Voice match: {best_ref["voice_match"]}/5')

## 9. Сводка RTF

In [ ]:
best_results = [r for r in results if r['ref_id']==BEST_REF_ID]

print('=== RTF для выбранного референса ===')
print(f'{"Фраза":<55} {"Синтез":<8} {"Аудио":<8} {"RTF":<6}')
print('-' * 80)
for r in best_results:
    print(f'{r["test_text"][:53]:<55} {r["synth_time"]:<8.1f} {r["audio_dur"]:<8.1f} {r["rtf"]:<6.2f}')

avg_rtf = sum(r['rtf'] for r in best_results) / len(best_results)
print(f'\nСредний RTF: {avg_rtf:.2f}')
if avg_rtf < 1.0:
    print('✅ real-time — XTTS можно юзать live на G1')
elif avg_rtf < 2.0:
    print('🟡 небольшая задержка — терпимо для демо')
else:
    print('🔴 медленно — генерить wav заранее, не live')

## 10. Сгенерировать пресеты (16 файлов)

In [ ]:
PRESETS = {
    'shtirlits_intro': 'Ну, значицца, Штирлиц... он вообще почти весь восемьдесят шестой по телевизору был, ну вы помните.',
    'shtirlits_joke': 'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
    'volodka_intro': 'А вот Вовочка... ну, который из школы... этот, как его... хулиган.',
    'volodka_joke': 'Учительница спрашивает Вовочку: какую оценку тебе поставить? Пять, Марьиванна! Ну, ты ещё посчитай. Ладно, четыре, но не меньше!',
    'rzhevsky_intro': 'Ну этот... поручик наш... Ржевский, да... он вообще к дамам постоянно лез.',
    'rzhevsky_joke': 'Поручик Ржевский на балу подходит к Наташе Ростовой: Наташа, а Наташа, давайте я вас поцелую! Поручик, я же дама! Ну, тем более!',
    'new_russians_intro': 'А это ещё тогда, в восемьдесят шестом, кооперативы пошли, ну, эти... в малиновых пиджаках.',
    'new_russians_joke': 'Встречаются два новых русских. Один говорит: Я вчера Запорожец купил! Ну и как? Крылья бабочки, малиновый цвет, всё дела!',
    'chapaev_intro': 'Ну, Чапаев, понятное дело... он вообще к нам из гражданской войны пришёл, но мы его любим.',
    'chapaev_joke': 'Чапаев спрашивает Петьку: Петька, рубль есть? Нет, Василий Иваныч. А два есть? Тоже нет. Ну вот, а ты говоришь — капиталистическая революция.',
    'coffee_intro': 'Угу, щас, Олег Тарасыч... кофеварку найду.',
    'coffee_obstacle': 'Бля, тут кто-то стоит... дай пройти.',
    'coffee_no_cup': 'Не вижу никакой чашки, Олег. Где кофе-то?',
    'coffee_got_it': 'Взял. Сейчас принесу.',
    'coffee_dropped': 'Ой... выронил, бля.',
    'coffee_done': 'Вот ваш кофе, Олег. Не обожгись, бля.',
}

os.makedirs('/content/presets', exist_ok=True)
preset_results = []

print(f'Генерация пресетов с референсом {BEST_REF_ID}...')
for name, text in PRESETS.items():
    out = f'/content/presets/{name}.wav'
    print(f'  → {name}')
    try:
        t0 = time.time()
        tts.tts_to_file(
            text=text,
            speaker_wav=BEST_REF_AUDIO,
            language='ru',
            file_path=out,
        )
        dt = time.time() - t0
        audio, sr_out = sf.read(out)
        dur = len(audio) / sr_out
        print(f'    ✅ {dur:.1f}сек, синтез {dt:.1f}с')
        preset_results.append({'name': name, 'duration': dur, 'synth_time': dt, 'path': out, 'text': text})
    except Exception as e:
        print(f'    ❌ {e}')

print(f'\n✅ {len(preset_results)}/{len(PRESETS)} пресетов сгенерировано')

## 11. Прослушать пресеты

In [ ]:
for r in preset_results:
    print(f'\n>>> {r["name"]} ({r["duration"]:.1f}сек)')
    print(f'    {r["text"]}')
    display(Audio(r['path']))

## 12. Архив для G1

In [ ]:
import json, shutil

manifest_out = {
    'model': 'Coqui XTTS v2',
    'language': 'ru',
    'reference_audio': best_ref['wav'],
    'reference_text': BEST_REF_TEXT,
    'reference_rating': best_ref['rating'],
    'reference_voice_match': best_ref['voice_match'],
    'reference_duration': best_ref['duration'],
    'reference_pitch': best_ref['pitch'],
    'dataset': 'burunov-voice/burunov-only-v2',
    'transcription_source': 'manual v2 (друг кореша, 2026-07-08)',
    'device': 'colab-T4',
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'avg_rtf': avg_rtf,
    'presets': [
        {'name': r['name'], 'file': f"{r['name']}.wav", 'duration_s': r['duration'], 'text': r['text']}
        for r in preset_results
    ],
}
with open('/content/presets/manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_out, f, ensure_ascii=False, indent=2)
print('✅ manifest.json сохранён')

shutil.copy(BEST_REF_AUDIO, f'/content/presets/{best_ref["wav"]}')
with open(f'/content/presets/reference_text.txt', 'w', encoding='utf-8') as f:
    f.write(BEST_REF_TEXT)

!cd /content && tar -czf burunov_presets.tar.gz presets/
!ls -lh /content/burunov_presets.tar.gz

print('\n📥 Скачай burunov_presets.tar.gz через панель файлов слева')